# 12 · Exact Answer Token (EAT) experiment — validate, *then* (separately) ship

**Idea.** Locate the verbatim span that actually answers the question ("Paris", "1969") and, for a flagged answer, point at *that* span. From Orgad et al., *LLMs Know More Than They Show* (ICLR 2025): probing the hidden state **at the exact answer token** detects errors far better than at the last token.

**Why it matters here.** Every HallKing detector currently anchors at an *end-of-sequence* token (SEP=second-last, TSV=last, HalluShift=whole range). That end-anchor is exactly the suspect behind two documented weaknesses: the weak SEP/HalluShift heads and the **length confound** (heads ride answer length — see `docs/head_audit.md`). Re-anchoring at the EAT is a new lever for both.

**Gate (do NOT skip).** The demo highlight (Phase 3) is only built *after* this notebook verifies. **2a** (extraction quality) is the first gate; **2b** (anchor vs AUROC) is the real test. Run in se_probes_env.

In [1]:
import os, sys, json
os.environ.setdefault('HF_HOME', r'D:/LLAMA CACHE/huggingface')
sys.path.insert(0, os.path.abspath(os.path.join('..', 'hallking')))
sys.path.insert(0, os.path.abspath(os.path.join('..', 'tools')))
import warnings; warnings.filterwarnings('ignore')
import numpy as np, pandas as pd, torch
from types import SimpleNamespace
from scipy.stats import spearmanr
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score
import eat, eat_eval, retrain, metrics as M
ROOT = os.path.abspath('..'); DATA = os.path.join(ROOT, 'data'); os.makedirs(DATA, exist_ok=True)
SEED = 42
print('torch', torch.__version__, '| cuda', torch.cuda.is_available())

torch 2.7.1+cu118 | cuda True


## Phase 2a · Extraction quality — the first gate (cheap, no retrain)
Generate a single-sentence answer per question, run the LLM self-extractor, and measure: extraction rate, EAT==gold (exact match), EAT↔gold (contains), and the failure breakdown. Per-row output is saved to `data/eat_eval_<ds>_n<N>.jsonl`.

**Read the gate like this:** a high *extraction rate* with a high *EAT↔gold among correct answers* means the extractor works. If extraction rate is low, or it frequently picks the wrong span when the answer is right (`wrong span` failures), STOP and fix the extractor before 2b.

In [2]:
DS_2A = ['triviaqa', 'squad']   # web_questions excluded (gold unusable)
N_2A  = 200                     # questions per dataset (GPU; a few minutes each)
rows_2a = {}
for ds in DS_2A:
    rows_2a[ds] = eat_eval.evaluate(dataset=ds, n=N_2A, save=True, verbose=True)

Using the latest cached version of the dataset since trivia_qa couldn't be found on the Hugging Face Hub
Found the latest cached dataset configuration 'rc' at D:\LLAMA CACHE\huggingface\datasets\trivia_qa\rc\0.0.0\0f7faf33a3908546c6fd5b73a660e0f8ff173c2f (last modified on Wed Mar 18 11:01:14 2026).


[eat_eval] triviaqa: 200 questions (offset=0)
[HallKing] loading meta-llama/Meta-Llama-3.1-8B-Instruct (4bit=True) ...


Loading weights: 100%|██████████| 291/291 [01:23<00:00,  3.47it/s]


[HallKing] model ready | num_layers=32 | device=cuda:0


[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer TokenizersBackend. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


  [0/200] A='Ross Bagdasarian Sr. was the man behind The Chipmunks.'  EAT='Ross Bagdasarian Sr.'
  [25/200] A="Men Against the Sea and Pitcairn's Island were two sequ"  EAT='the famous novel "Mutiny on the Bounty".'
  [50/200] A='The Crow.'  EAT='The Crow.'
  [75/200] A='The 70s American show "All in the Family" was based on '  EAT='All in the Family'
  [100/200] A='The Oscar-nominated film "The Full Monty" had "You Sexy'  EAT='The Full Monty'
  [125/200] A='Pete Sampras was seeded 12th when he won his first US O'  EAT='12th'
  [150/200] A='Frank Harris wrote My Life and Loves in 1926, originall'  EAT='Frank Harris'
  [175/200] A='Phil Collins recorded on the record labels Atlantic Rec'  EAT='Atlantic Records'
[HallKing] model unloaded.

[eat_eval] triviaqa  (n=200)
  extraction rate (usable verbatim span) :  95.5%  (191/200)
  answer contains gold (model correct)    :  70.0%  (140/200)
  EAT == gold (exact match)               :  49.5%  (99/200)
  EAT <-> gold (contains, either way)  

Using the latest cached version of the dataset since squad couldn't be found on the Hugging Face Hub
Found the latest cached dataset configuration 'plain_text' at D:\LLAMA CACHE\huggingface\datasets\squad\plain_text\0.0.0\7b6d24c440a36b6815f21b70d25016731768db1f (last modified on Mon Mar 30 15:38:13 2026).


[eat_eval] squad: 200 questions (offset=0)
[HallKing] loading meta-llama/Meta-Llama-3.1-8B-Instruct (4bit=True) ...


Loading weights: 100%|██████████| 291/291 [01:00<00:00,  4.77it/s]


[HallKing] model ready | num_layers=32 | device=cuda:0
  [0/200] A='The Denver Broncos represented the AFC at Super Bowl 50'  EAT='The Denver Broncos'
  [25/200] A='Super Bowl 50 determined the NFL champion for the 2015 '  EAT='2015'
  [50/200] A='The New England Patriots.'  EAT='The New England Patriots.'
  [75/200] A='Von Miller is a linebacker.'  EAT='linebacker'
  [100/200] A='The special guests for the Super Bowl halftime show var'  EAT='The Weeknd'
  [125/200] A='The NFL decided between the Los Angeles Memorial Colise'  EAT=None
  [150/200] A='The top two stadium choices for Super Bowl 50 were anno'  EAT='May 21, 2013'
  [175/200] A="Levi's Stadium opened on April 28, 2014."  EAT='April 28, 2014.'
[HallKing] model unloaded.

[eat_eval] squad  (n=200)
  extraction rate (usable verbatim span) :  88.0%  (176/200)
  answer contains gold (model correct)    :  42.5%  (85/200)
  EAT == gold (exact match)               :  33.5%  (67/200)
  EAT <-> gold (contains, either way)     :  40.0%

### 2a · Inspect examples + failure cases
Eyeball that the extracted span is the actual answer phrase, and look at where it goes wrong (model paraphrased / no clean answer / span landed on the wrong token).

In [3]:
def show(ds, k=12):
    rs = rows_2a[ds]
    print(f'=== {ds}: {k} examples ===')
    for r in rs[:k]:
        tag = 'OK ' if r['eat_contains_gold'] else ('x  ' if r['extracted'] else '-- ')
        print(f"  {tag} EAT={str(r['eat'])[:30]!r:32} gold0={str(r['refs'][0])[:24]!r:26} | A={r['answer'][:60]!r}")
    bad = [r for r in rs if r['answer_contains_gold'] and not r['eat_contains_gold']]
    print(f'\n  -- {len(bad)} WRONG-SPAN cases (answer was correct but EAT missed it) --')
    for r in bad[:8]:
        print(f"     EAT={str(r['eat'])[:30]!r:32} raw={str(r['raw_eat'])[:30]!r:32} | A={r['answer'][:55]!r}")
for ds in DS_2A:
    show(ds); print()

=== triviaqa: 12 examples ===
  x   EAT='Ross Bagdasarian Sr.'           gold0='David Seville'            | A='Ross Bagdasarian Sr. was the man behind The Chipmunks.'
  OK  EAT='Sunset Boulevard'               gold0='Sunset Blvd'              | A='Sunset Boulevard premiered in the US on 10th December 1993.'
  x   EAT='Herbert Henry Asquith'          gold0='Sir Henry Campbell-Banne' | A='Herbert Henry Asquith became the next British Prime Minister'
  OK  EAT='Exile'                          gold0='Internal exile'           | A='Exile had a 1978 No 1 hit with Kiss You All Over.'
  OK  EAT='breast cancer'                  gold0='Cancer pathology'         | A='Kathleen Ferrier died of breast cancer.'
  OK  EAT='Octopussy'                      gold0='Kamal kahn'               | A='Rita Coolidge sang the title song for the Bond film "Octopus'
  x   EAT='Mississippi'                    gold0='Utah (State)'             | A='Mississippi was the last US state to reintroduce alcohol aft'
  x   EA

### 2a · GATE decision
Rough bar to proceed to 2b: extraction rate ≳ 85%, and among answers that contain the gold, EAT↔gold ≳ 85% (few wrong-span failures). Note the numbers here in `docs/eat_audit.md`. **If this fails, stop — do not run 2b and do not touch the demo.**

## Phase 2b · Anchor comparison — the real test (GPU, gated)
> ⚠️ **Untested locally** (no GPU in the build env). Run only after 2a passes. Set `RUN_2B=True`.

One Instruct (fp16) pass caches, per question, **both** anchorings of every head plus a hallucination label, then we retrain the two probe heads on each anchoring and compare:
- **SEP** — all-layer hidden stack at the **EAT token** vs the **second-last** token (`_slt_stack`).
- **HalluShift** — 71-d features over the **EAT step window** vs the **whole** range (`_features_from_gen`).
- **TSV** — frozen centroid margin read at the **EAT token** vs the **last** token (`score_sentences`).

We report held-out AUROC (end vs EAT) and each head's Spearman correlation with answer length (the confound — lower |ρ| is better). Labels via the same hybrid judge used by `retrain.gen_and_cache`.

In [4]:
RUN_2B   = True                # <-- flip to True on the GPU box, after 2a passes
DS_2B    = 'triviaqa'           # cache+retrain dataset
N_2B     = 400
OFFSET_2B = 0
MAXTOK_2B = 64
CACHE_2B = os.path.join(DATA, f'eat_anchor_{DS_2B}_n{N_2B}.npz')
print('RUN_2B =', RUN_2B, '| cache ->', os.path.relpath(CACHE_2B, ROOT))

RUN_2B = True | cache -> data\eat_anchor_triviaqa_n400.npz


### 2b · Cache both anchorings (GPU). 
Reuses `eat.locate_eat_span` + `eat.eat_token_range` (unit-tested) to find the EAT token, then reads each head at the EAT anchor and at its usual end anchor from the SAME generation. SEP vectors are stored fp16 (135168-d × N × 2).

In [5]:
def _eat_step_window(eng, gen, answer, eat_text, min_steps=2):
    '''Return (eat_abs_idx, (a_step, b_step)) for HalluShift's step-range view, or None.'''
    span = eat.locate_eat_span(answer, eat_text) if eat_text else None
    if span is None:
        return None
    t0, t1 = eat.eat_token_range(eng, gen, span[0], span[1])      # absolute token indices
    plen = gen['prompt_len']
    a, b = t0 - plen, t1 - plen                                   # -> generation-step indices
    while b - a < min_steps:                                      # widen so HalluShift doesn't fall back
        a = max(0, a - 1); b = b + 1
    return t1 - 1, (max(0, a), b)                                 # anchor = last EAT token

def cache_2b():
    from sep_adapter import SEPAdapter
    from hallushift_adapter import HalluShiftAdapter
    from run_dataset import load_qa, INSTRUCT_MODEL
    from engine import HallKingEngine
    from claim_label import label_hybrid
    qs, refs = load_qa(DS_2B, n=N_2B, offset=OFFSET_2B)
    eng = HallKingEngine(model_name=INSTRUCT_MODEL, fp16_nonquant=True).load()
    sep = SEPAdapter(eng); hs = HalluShiftAdapter(eng); hs.num_layers = eng.num_layers
    keep_q, keep_r, answers = [], [], []
    sep_end, sep_eat, hs_end, hs_eat, ans_len, has_eat, eat_end = [], [], [], [], [], [], []
    from tqdm.auto import tqdm
    for q, rf in tqdm(list(zip(qs, refs)), desc=f'EAT cache ({DS_2B})', unit='q'):
        gen = eng.generate_sentence(q, max_new_tokens=MAXTOK_2B)
        ans = gen['answer_full'].strip()
        if not ans:
            continue
        eat_txt = eat.extract_eat_text(eng, q, ans)
        sp_eat = eat.locate_eat_span(ans, eat_txt) if eat_txt else None
        win = _eat_step_window(eng, gen, ans, eat_txt)
        seq = gen['sequences']; plen = gen['prompt_len']
        slt = max(seq.shape[1] - 2, plen)
        H = eng.forward_hidden_states(seq)                        # one clean forward (SEP)
        e_idx = win[0] if win else slt                            # fall back to SLT when no EAT
        sep_end.append(sep._slt_stack(H, slt).astype(np.float16).reshape(-1))
        sep_eat.append(sep._slt_stack(H, e_idx).astype(np.float16).reshape(-1))
        hs_end.append(np.asarray(hs.features(gen), np.float64))
        if win:
            a, b = win[1]; go = gen['gen_output']
            view = SimpleNamespace(hidden_states=go.hidden_states[a:b],
                                   attentions=go.attentions[a:b], logits=go.logits[a:b])
            try:
                hs_eat.append(np.asarray(hs._features_from_gen(view), np.float64))
            except Exception:
                hs_eat.append(np.asarray(hs.features(gen), np.float64))
        else:
            hs_eat.append(np.asarray(hs.features(gen), np.float64))
        keep_q.append(q); keep_r.append(rf); answers.append(ans)
        ans_len.append(len(ans)); has_eat.append(win is not None)
        eat_end.append(sp_eat[1] if sp_eat else -1)   # EAT end char offset (reused by the TSV cell)
    print(f'cached {len(answers)} rows | EAT found for {sum(has_eat)} ({100*np.mean(has_eat):.0f}%)')
    labels, _ = label_hybrid(keep_q, answers, keep_r, eng, verbose=True)
    eng.unload()
    y = np.asarray(labels).astype(int)
    np.savez_compressed(CACHE_2B, y=y, ans_len=np.asarray(ans_len), has_eat=np.asarray(has_eat),
                        eat_end=np.asarray(eat_end),
                        sep_end=np.stack(sep_end), sep_eat=np.stack(sep_eat),
                        hs_end=np.stack(hs_end), hs_eat=np.stack(hs_eat),
                        question=np.array(keep_q, dtype=object), answer=np.array(answers, dtype=object))
    print('saved', os.path.relpath(CACHE_2B, ROOT))

if RUN_2B and not os.path.exists(CACHE_2B):
    cache_2b()
else:
    print('skip cache:', 'RUN_2B=False' if not RUN_2B else 'cache exists')

Using the latest cached version of the dataset since trivia_qa couldn't be found on the Hugging Face Hub
Found the latest cached dataset configuration 'rc' at D:\LLAMA CACHE\huggingface\datasets\trivia_qa\rc\0.0.0\0f7faf33a3908546c6fd5b73a660e0f8ff173c2f (last modified on Wed Mar 18 11:01:14 2026).


[HallKing] loading meta-llama/Meta-Llama-3.1-8B-Instruct (4bit=True) ...


[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|██████████| 291/291 [01:02<00:00,  4.65it/s]


[HallKing] model ready | num_layers=32 | device=cuda:0


EAT cache (triviaqa): 100%|██████████| 400/400 [21:30<00:00,  3.23s/q]

cached 400 rows | EAT found for 378 (94%)
  [hybrid] 295 truthful by substring; judging 105 non-match rows ...



QA-judge: 100%|██████████| 105/105 [00:41<00:00,  2.50claim/s]


[HallKing] model unloaded.
saved data\eat_anchor_triviaqa_n400.npz


### 2b · Retrain the probe heads on each anchor + compare AUROC
Shared 75/25 split. SEP probes via `retrain.retrain_sep`; HalluShift via `retrain.retrain_hallushift`. Each head trained twice (end-anchored vs EAT-anchored), evaluated on the same held-out rows.

In [6]:
def auroc(y, s): return roc_auc_score(y, s) if len(set(y)) > 1 else float('nan')
if RUN_2B:
    Z = np.load(CACHE_2B, allow_pickle=True)
    y, ans_len = Z['y'], Z['ans_len']
    tr, te = train_test_split(np.arange(len(y)), test_size=0.25, stratify=y, random_state=SEED)
    def sep_auc(feats):
        probe = retrain.retrain_sep(feats[tr].astype(np.float32), y[tr])[0]
        Xb = feats.astype(np.float32)
        s = probe['s_bmodel'].predict_proba(Xb[te])[:, 1]
        return auroc(y[te], s), s
    sep_end_auc, sep_end_s = sep_auc(Z['sep_end'])
    sep_eat_auc, sep_eat_s = sep_auc(Z['sep_eat'])
    def hs_auc(feats):
        df = pd.DataFrame({f'hs_feat_{j:02d}': feats[:, j] for j in range(feats.shape[1])})
        state, scaler = retrain.retrain_hallushift(df.iloc[tr], y[tr])
        from classifier import CombinedNN
        m = CombinedNN(32); m.load_state_dict(state); m.eval()
        with torch.no_grad():
            s = torch.sigmoid(m(torch.tensor(scaler.transform(feats), dtype=torch.float32))).numpy().ravel()
        return auroc(y[te], s[te]), s[te]
    hs_end_auc, _ = hs_auc(Z['hs_end'])
    hs_eat_auc, _ = hs_auc(Z['hs_eat'])
    print(f'{"head":12} {"END anchor":>11} {"EAT anchor":>11}')
    print(f'{"SEP":12} {sep_end_auc:11.3f} {sep_eat_auc:11.3f}')
    print(f'{"HalluShift":12} {hs_end_auc:11.3f} {hs_eat_auc:11.3f}')
else:
    print('RUN_2B=False')

  [HalluShift retrain] best val AUROC=0.8665
  [HalluShift retrain] best val AUROC=0.8326
head          END anchor  EAT anchor
SEP                0.863       0.767
HalluShift         0.576       0.731


### 2b · TSV (frozen) end vs EAT read
No retrain — just move the read position. `score_qa` reads the last token; `score_sentences(.,.,[eat_end])` reads the EAT token of the steered forward. Uses the deployed sentence TSV checkpoint.

In [7]:
if RUN_2B:
    from engine import HallKingEngine
    from run_dataset import INSTRUCT_MODEL
    from tsv_adapter import TSVAdapter
    Z = np.load(CACHE_2B, allow_pickle=True); qs = Z['question']; ans = Z['answer']; ee = Z['eat_end']
    ckpt = os.path.join(ROOT, 'artifacts', 'tsv', 'best_checkpoint_sentence_s1.pt')
    eng = HallKingEngine(model_name=INSTRUCT_MODEL, fp16_nonquant=True).load()
    tsv = TSVAdapter(eng, ckpt_path=ckpt).load()
    tsv_end, tsv_eat = [], []
    from tqdm.auto import tqdm
    for i, (q, a) in enumerate(tqdm(list(zip(qs, ans)), desc='TSV end/EAT', unit='q')):
        m_end = tsv.score_qa(q, a)['tsv_margin']   # last-token read
        tsv_end.append(m_end)
        # reuse the cached EAT span (same span SEP/HalluShift anchored on); fall back to last token
        tsv_eat.append(tsv.score_sentences(q, a, [int(ee[i])])[0] if ee[i] >= 0 else m_end)
    eng.unload()
    print(f'TSV       END={auroc(y[te], np.asarray(tsv_end)[te]):.3f}  EAT={auroc(y[te], np.asarray(tsv_eat)[te]):.3f}')
else:
    print('RUN_2B=False')

[HallKing] loading meta-llama/Meta-Llama-3.1-8B-Instruct (4bit=True) ...


Loading weights: 100%|██████████| 291/291 [01:42<00:00,  2.85it/s]


[HallKing] model ready | num_layers=32 | device=cuda:0
[TSV] loaded checkpoint | layer=9 lam=5.0 cos_temp=0.1 | trained AUROC=0.8165


TSV end/EAT: 100%|██████████| 400/400 [01:44<00:00,  3.81q/s]


[HallKing] model unloaded.
TSV       END=0.884  EAT=0.823


### 2b · Length-confound diagnostic
Spearman ρ between each head's score and answer length, end vs EAT. The hypothesis: anchoring at the answer token (not the sequence end, whose hidden state drifts with position/length) **shrinks |ρ|**. Compare against `docs/head_audit.md`'s end-anchored numbers.

In [8]:
if RUN_2B:
    def rho(s): return spearmanr(s, ans_len[te]).correlation
    print('Spearman |rho| with answer length (lower is better):')
    print(f'  SEP        end={rho(sep_end_s):+.3f}  EAT={rho(sep_eat_s):+.3f}')
    # (HalluShift/TSV: recompute full-vector scores above if you want their rho too)
else:
    print('RUN_2B=False')

Spearman |rho| with answer length (lower is better):
  SEP        end=+0.364  EAT=+0.434


## Verdict → `docs/eat_audit.md`
Fill the table with the 2a gate numbers and the 2b end-vs-EAT AUROC + length-ρ. **Demo (Phase 3) proceeds only if** 2a passed AND EAT anchoring helps (or at least matches without misleading). If EAT ≈ END on AUROC but the highlight is still wanted as pure UX, that is allowed — but it must be labelled 'exact answer phrase', not 'the hallucinated token' (the flag stays sentence-level).